# SLM-RL-Agents Tutorial

This notebook provides a step-by-step guide to training Small Language Model (SLM) agents using Reinforcement Learning from Human Feedback (RLHF).

## Overview

We'll cover the complete RLHF pipeline:

1. **Data Preparation**: Loading and formatting datasets
2. **Supervised Fine-Tuning (SFT)**: Teaching the model to follow instructions
3. **Reward Model Training**: Learning human preferences
4. **PPO/DPO Training**: Optimizing the policy with learned rewards
5. **Evaluation**: Measuring model quality
6. **Agent Deployment**: Using the trained model

By the end of this tutorial, you'll have a working SLM agent trained with RLHF!

## Setup

First, let's install dependencies and import required libraries.

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install -e ".[all]"

In [ ]:
import os
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

# Check GPU availability
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 1. Configuration

Let's set up our training configuration. We'll use Pythia-160M as our base model since it's small enough to train quickly but large enough to learn useful behaviors.

In [ ]:
# Configuration
CONFIG = {
    # Model
    "model_name": "EleutherAI/pythia-160m-deduped",
    
    # Directories
    "data_dir": "./data",
    "output_dir": "./outputs/tutorial",
    
    # Training parameters (reduced for quick demo)
    "sft_epochs": 1,
    "batch_size": 4,
    "max_samples": 1000,  # Small subset for demo
    "max_seq_length": 512,
    
    # LoRA parameters
    "lora_r": 8,
    "lora_alpha": 16,
}

# Create directories
os.makedirs(CONFIG["data_dir"], exist_ok=True)
os.makedirs(CONFIG["output_dir"], exist_ok=True)

print("Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

## 2. Load and Prepare Data

We'll use a small subset of UltraChat for SFT and UltraFeedback for preference training.

In [ ]:
# পূর্বে: from src.data import load_sft_dataset, load_preference_dataset
from src.slm_rl_agent.data import load_sft_dataset, load_preference_dataset

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer loaded: vocab_size={tokenizer.vocab_size}")

In [ ]:
# Load SFT dataset
print("Loading SFT dataset...")
sft_dataset = load_sft_dataset(
    "HuggingFaceH4/ultrachat_200k",
    tokenizer=tokenizer,
    max_samples=CONFIG["max_samples"],
)
print(f"SFT dataset: {len(sft_dataset)} examples")
print(f"Sample: {sft_dataset[0]['text'][:200]}...")

In [ ]:
# Load preference dataset
print("\nLoading preference dataset...")
pref_dataset = load_preference_dataset(
    "HuggingFaceH4/ultrafeedback_binarized",
    tokenizer=tokenizer,
    max_samples=CONFIG["max_samples"],
)
print(f"Preference dataset: {len(pref_dataset)} examples")
print(f"Columns: {pref_dataset.column_names}")

## 3. Supervised Fine-Tuning (SFT)

SFT is the first stage of RLHF. We train the base model to follow instructions using demonstration data. This creates a solid foundation for the subsequent RL stages.

In [ ]:
# পূর্বে: from src.training import SFTTrainerWrapper
#         from src.training.sft_trainer import SFTTrainingConfig
from src.slm_rl_agent.rl import SFTTrainerWrapper
from src.slm_rl_agent.rl.sft_trainer import SFTTrainingConfig

# Configure SFT training
sft_config = SFTTrainingConfig(
    model_name_or_path=CONFIG["model_name"],
    output_dir=f"{CONFIG['output_dir']}/sft",
    num_train_epochs=CONFIG["sft_epochs"],
    per_device_train_batch_size=CONFIG["batch_size"],
    max_seq_length=CONFIG["max_seq_length"],
    use_lora=True,
    lora_r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    use_quantization=True,
    logging_steps=10,
    save_steps=100,
)

print("SFT Configuration ready!")

In [ ]:
# Initialize and run SFT training
sft_trainer = SFTTrainerWrapper(sft_config)

print("Starting SFT training...")
print("This may take a few minutes depending on your hardware.")

# Train
sft_metrics = sft_trainer.train(sft_dataset)

# Save the model
sft_model_path = f"{CONFIG['output_dir']}/sft/final"
sft_trainer.save_model(sft_model_path)

print(f"\nSFT training complete!")
print(f"Model saved to: {sft_model_path}")
print(f"Training loss: {sft_metrics.get('train_loss', 'N/A')}")

## 4. Test the SFT Model

Let's test our SFT model to see how it performs before RL training.

In [ ]:
from src.agent import SLMAgent

# Load the SFT model
sft_agent = SLMAgent.from_pretrained(sft_model_path)

# Test prompts
test_prompts = [
    "What is machine learning?",
    "Explain the difference between Python and JavaScript.",
    "Write a short poem about coding.",
]

print("Testing SFT model:\n")
for prompt in test_prompts:
    print(f"Prompt: {prompt}")
    response = sft_agent.generate(prompt, max_new_tokens=100, temperature=0.7)
    print(f"Response: {response}")
    print("-" * 50)

## 5. DPO Training (Simpler Alternative to PPO)

Direct Preference Optimization (DPO) is a simpler alternative to PPO that doesn't require a separate reward model. It directly optimizes the policy using preference data.

In [ ]:
# পূর্বে: from src.training import DPOTrainerWrapper
#         from src.training.dpo_trainer import DPOTrainingConfig
from src.slm_rl_agent.rl import DPOTrainerWrapper
from src.slm_rl_agent.rl.dpo_trainer import DPOTrainingConfig

# Configure DPO training
dpo_config = DPOTrainingConfig(
    model_name_or_path=sft_model_path,
    output_dir=f"{CONFIG['output_dir']}/dpo",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    beta=0.1,  # KL penalty coefficient
    learning_rate=5e-7,
    max_seq_length=CONFIG["max_seq_length"],
    use_lora=True,
    lora_r=CONFIG["lora_r"],
    logging_steps=10,
    save_steps=50,
)

print("DPO Configuration ready!")

In [ ]:
# Initialize and run DPO training
dpo_trainer = DPOTrainerWrapper(dpo_config)

print("Starting DPO training...")

# Train
dpo_metrics = dpo_trainer.train(pref_dataset)

# Save
dpo_model_path = f"{CONFIG['output_dir']}/dpo/final"
dpo_trainer.save_model(dpo_model_path)

print(f"\nDPO training complete!")
print(f"Model saved to: {dpo_model_path}")

## 6. Evaluation

Let's evaluate our trained model using various metrics.

In [ ]:
from src.evaluation import EvaluationSuite, compute_perplexity, compute_distinct_n

# Load the DPO model for evaluation
dpo_agent = SLMAgent.from_pretrained(dpo_model_path)

# Generate responses for evaluation
eval_prompts = [
    "Explain quantum computing.",
    "What are the benefits of exercise?",
    "How does photosynthesis work?",
    "Describe the water cycle.",
    "What is artificial intelligence?",
]

print("Generating responses for evaluation...")
responses = []
for prompt in eval_prompts:
    response = dpo_agent.generate(prompt, max_new_tokens=150, temperature=0.7)
    responses.append(response)
    print(f"\nPrompt: {prompt}")
    print(f"Response: {response[:200]}...")

In [ ]:
# Compute diversity metrics
diversity_metrics = compute_distinct_n(responses, n_values=[1, 2, 3])

print("\nDiversity Metrics:")
for metric, value in diversity_metrics.items():
    print(f"  {metric}: {value:.4f}")

## 7. Compare SFT vs DPO

Let's compare the outputs of our SFT and DPO models to see the improvement.

In [ ]:
comparison_prompts = [
    "What's the best way to learn programming?",
    "Explain climate change to a child.",
]

print("Model Comparison:\n")
print("=" * 80)

for prompt in comparison_prompts:
    print(f"\nPrompt: {prompt}\n")
    
    sft_response = sft_agent.generate(prompt, max_new_tokens=150, temperature=0.7)
    dpo_response = dpo_agent.generate(prompt, max_new_tokens=150, temperature=0.7)
    
    print(f"SFT Model:\n{sft_response}\n")
    print(f"DPO Model:\n{dpo_response}")
    print("=" * 80)

## 8. Using the Trained Agent

Now let's explore different ways to use our trained agent.

In [ ]:
# Interactive conversation mode
agent = SLMAgent.from_pretrained(dpo_model_path)

# Start a conversation
agent.start_conversation()

# Multi-turn conversation
turns = [
    "Hi! I want to learn about space.",
    "What's the closest planet to Earth?",
    "How far away is it?",
]

print("Multi-turn Conversation:\n")
for turn in turns:
    print(f"User: {turn}")
    response = agent.generate(turn, max_new_tokens=100)
    print(f"Agent: {response}\n")

# End conversation and get history
history = agent.end_conversation()
print(f"Conversation had {len(history)} messages.")

## 9. Save and Export

Finally, let's save our model for deployment.

In [ ]:
# Get model info
model_info = agent.get_model_info()

print("Final Model Information:")
for key, value in model_info.items():
    print(f"  {key}: {value}")

print(f"\nModel saved at: {dpo_model_path}")
print("\nTo use this model:")
print(f"  agent = SLMAgent.from_pretrained('{dpo_model_path}')")
print("  response = agent.generate('Your prompt here')")

## Summary

In this tutorial, we:

1. Set up our environment and loaded datasets
2. Trained an SFT model to follow instructions
3. Applied DPO to align the model with human preferences
4. Evaluated the model using diversity metrics
5. Compared SFT vs DPO outputs
6. Demonstrated multi-turn conversations

### Next Steps

- Try different base models (SmolLM2, Pythia-410M)
- Experiment with PPO instead of DPO
- Train a reward model for more control
- Fine-tune hyperparameters for better results
- Deploy the model using the inference server

For more details, check out the README and the full documentation!